# Capstone demo

Повний mini-research workflow: calibration → optimization → verification → sensitivity → uncertainty → reproducibility.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from model import fit_parameters, InterventionConfig, optimize_intervention, grid_verify, budget_sensitivity, bootstrap_optimized_outcomes, terminal_state, state_trajectory, effective_params

## 1. Research question and data

In [ ]:
cfg = json.loads((ROOT/'experiment_config.json').read_text(encoding='utf-8'))
obs = pd.read_csv(ROOT/'data'/'observations.csv')
print(cfg['research_question'])
obs.head()

## 2. Calibration

In [ ]:
fit = fit_parameters(obs.time, obs.observed_state, s0=cfg['model']['s0'])
p = fit['params']
print({'q_hat':p.q, 'k_hat':p.k, 'rmse':fit['rmse']})

In [ ]:
tt=np.linspace(obs.time.min(), obs.time.max(), 250)
plt.figure(figsize=(8,4.5))
plt.scatter(obs.time, obs.observed_state, label='observations')
plt.plot(tt, state_trajectory(tt,p), label='calibrated model')
plt.xlabel('time'); plt.ylabel('state'); plt.legend(); plt.tight_layout(); plt.show()

## 3. Constrained intervention optimization

In [ ]:
ic = InterventionConfig(**cfg['intervention'])
base = terminal_state(p,0,0,ic)
opt = optimize_intervention(p,ic)
print('baseline terminal:', base)
print(opt)

## 4. Independent verification

In [ ]:
grid = grid_verify(p,ic,resolution=cfg['verification']['grid_resolution'])
print(grid)
print('optimizer-grid gap:', opt['terminal_state']-grid['terminal_state'])

## 5. Budget sensitivity

In [ ]:
sens = budget_sensitivity(p,ic,cfg['sensitivity']['budgets'])
sens

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(sens.budget, sens.terminal_state, marker='o')
plt.xlabel('budget'); plt.ylabel('optimized terminal state'); plt.tight_layout(); plt.show()

## 6. Bootstrap uncertainty

In [ ]:
boot = bootstrap_optimized_outcomes(obs.time, obs.observed_state, ic, n_boot=cfg['uncertainty']['n_boot'], seed=cfg['uncertainty']['seed'], s0=cfg['model']['s0'])
boot[['q_hat','k_hat','terminal_state']].quantile([0.025,0.5,0.975])

## 7. Research conclusion prompts

- Чи підтримує computational experiment гіпотезу?
- Наскільки optimizer узгоджується з незалежною verification?
- Що змінюється при зміні budget?
- Яка uncertainty оптимізованого результату?
- Які висновки стосуються лише моделі, а не реальної системи?